## Imports Section:

In [1]:
import numpy as np
import pandas as pd
import random
import scipy
from scipy import stats
import datetime as dt
import dask.dataframe as dd

In [2]:
import librosa
import warnings

In [3]:
import glob
import matplotlib.pyplot as plt
from matplotlib import colors
import soundfile as sf
import matplotlib.patches as patches
from pathlib import Path

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

In [5]:
import sys

# append the path of the
# parent directory
sys.path.append('..')
sys.path.append('../src/')
sys.path.append('../src/models/bat_call_detector/batdetect2/')

import src.batdt2_pipeline as batdetect2_pipeline
import src.file_dealer as file_dealer

In [6]:
from sklearn.cluster import KMeans
LABEL_FOR_GROUPS = {
                    0: 'LF', 
                    1: 'HF'
                    }

## Function Definitions (most important functions are in `src/`)
### Below are the functions used for plotting and visualization

In [7]:
def convert_bd2df_ravenpro(df: pd.DataFrame):
    """
    Converts a dataframe to the format used by RavenPro
    """

    ravenpro_df = df.copy()

    ravenpro_df.rename(columns={
        "start_time": "Begin Time (s)",
        "end_time": "End Time (s)",
        "low_freq": "Low Freq (Hz)",
        "high_freq": "High Freq (Hz)",
        "event": "Annotation",
        "freq_group":"Manually-Verified Phonic Group",
        'snr_nist_quick_dB':'SNR NIST Quick (dB)',
        'delta_time_s':'Delta Time (s)'
    }, inplace=True)

    ravenpro_df["Selection"] = np.arange(0, df.shape[0]).astype('int') + 1
    ravenpro_df["View"] = "Waveform 1"
    ravenpro_df["Channel"] = "1"

    return ravenpro_df

In [8]:
FREQ_COLORS = {'LF':'cyan', 'HF':'orange'}

In [9]:
file_sites = {'20220730_053000':'Carp',
 '20220826_070000':'Central',
 '20220727_083000':'Foliage',
 '20220829_090000':'Foliage'}
bd2_dets_save_dir = Path(f'20250127__group_threshold_sweep_results')
raventxt_human_files_dir = f'{Path.home()}/Documents/Research/mila_files/mila-human-wav-txt'
kaleidoscope_dets_save_dir = Path.home() / f'Documents'
extension = ".txt"
sep = "\t"

SITE_NAMES = {'Carp':'Carp Pond', 'Foliage':'Foliage', 'Central':'Central Pond'}
TP_CLASSIFICATION_THRESHOLD = 0.010
OVERLAP_TIME_THRESHOLD = 0.012

file_keys = list(file_sites.keys())
file_keys

['20220730_053000', '20220826_070000', '20220727_083000', '20220829_090000']

## Following the confusion matrix structure shown below

![image](example_confusion_matrix.png)

In [10]:
def return_confusion_matrix_from_comparing_two_detectors(human_df, machine_df):
    dist_matrix = np.zeros((len(machine_df), len(human_df)))
    association_matrix = np.zeros((len(machine_df), len(human_df)), dtype='bool')
    for index in range(len(machine_df)):
        batdetect2_row = machine_df.iloc[index]
        dist_to_all_calls = ((human_df['peak_frequency_time'] - batdetect2_row['peak_frequency_time_SPECTROGRAM']).values)
        dist_matrix[index,:] = dist_to_all_calls
    
        dist_to_all_calls[np.abs(dist_to_all_calls)>=TP_CLASSIFICATION_THRESHOLD] = 1
        dist_to_all_calls[np.abs(dist_to_all_calls)<TP_CLASSIFICATION_THRESHOLD] = 0
        association_matrix[index,:] = ~dist_to_all_calls.astype('bool')

    human_bd2_true_positives = human_df.loc[np.logical_or.reduce(association_matrix, axis=0)]
    bd2_false_negatives = human_df.loc[~(np.logical_or.reduce(association_matrix, axis=0))]
    bd2_true_positives = machine_df.loc[np.logical_or.reduce(association_matrix, axis=1)]
    bd2_false_positives = machine_df.loc[~(np.logical_or.reduce(association_matrix, axis=1))]

    return {'true_positives':len(bd2_true_positives), 
            'false_positives':len(bd2_false_positives), 
            'false_negatives':len(bd2_false_negatives), 
            'true_negatives':0}

def get_precision_and_recall_from_metrics(true_positives, false_positives, false_negatives):
    denom_precision = (true_positives + false_positives)
    denom_recall = (true_positives + false_negatives)
    if (denom_precision>0):
        precision = true_positives / denom_precision
    else:
        precision = np.NaN
    if (denom_recall>0):
        recall = true_positives / denom_recall
    else:
        recall = np.NaN
    
    return precision, recall

def gather_evaluation_results_between_bd2_and_human(bd2_human_df, batdetect2_df_thresh):
    file_batdetect2_cf = return_confusion_matrix_from_comparing_two_detectors(bd2_human_df, batdetect2_df_thresh)
    precision, recall = get_precision_and_recall_from_metrics(file_batdetect2_cf['true_positives'], 
                                                              file_batdetect2_cf['false_positives'], 
                                                              file_batdetect2_cf['false_negatives'])

    return file_batdetect2_cf, precision, recall

def apply_SNR_threshold_on_both_sets(bd2_human_df, batdetect2_df, SNR_thresh):
    bd2_human_df_SNR = bd2_human_df[bd2_human_df['adityas_method_snr_dB']>=SNR_thresh].copy()
    batdetect2_df_SNR = batdetect2_df[batdetect2_df['SNR']>=SNR_thresh].copy()

    return bd2_human_df_SNR, batdetect2_df_SNR

def make_new_row_in_eval_df(file_batdetect2_cf, precision, recall):
    row = pd.DataFrame([file_batdetect2_cf])
    row['precision'] = precision
    row['recall'] = recall

    return row

def generate_evaluation_df(bd2_human_df, batdetect2_df):
    batdetect2_eval = pd.DataFrame()

    snr_increment = 1.0
    snr_thresholds = np.arange(0, 20, snr_increment)
    for snr_thresh in snr_thresholds:
        bd2_human_df_SNR, batdetect2_df_SNR = apply_SNR_threshold_on_both_sets(bd2_human_df, batdetect2_df, snr_thresh)

        detthresh_increment = 0.01
        detection_thresholds = np.arange(0.01, 0.6+detthresh_increment, detthresh_increment)
        for det_thresh in detection_thresholds:
            det_thresh = round(det_thresh, 2)
            print(snr_thresh, det_thresh)
            batdetect2_df_SNR_detthresh = batdetect2_df_SNR[batdetect2_df_SNR['det_prob']>=det_thresh].copy()
            file_batdetect2_cf, precision, recall = gather_evaluation_results_between_bd2_and_human(bd2_human_df_SNR, 
                                                                                                    batdetect2_df_SNR_detthresh)
            row = make_new_row_in_eval_df(file_batdetect2_cf, precision, recall)
            row.insert(0, 'num_human_annotations', [len(bd2_human_df_SNR)])
            row.insert(0, 'num_bd2_detections', [len(batdetect2_df_SNR_detthresh)])
            row.insert(0, 'detection_threshold', [det_thresh])
            row.insert(0, 'SNR_threshold', [snr_thresh])
            batdetect2_eval = pd.concat([batdetect2_eval, row])
            assert(len(batdetect2_df_SNR_detthresh['freq_group'].unique())<=2)

    batdetect2_eval.reset_index(drop=True, inplace=True)
    return batdetect2_eval

In [ ]:
batdetect2_eval = pd.DataFrame()
batdetect2_eval_LF = pd.DataFrame()
batdetect2_eval_HF = pd.DataFrame()
for file_key in file_keys:
    wav_filename = file_key
    site = file_sites[file_key]
    plot_file = Path(f'{raventxt_human_files_dir}/{wav_filename}.WAV')
    raventxt_human_filename = f'{wav_filename}_manually_verified_by_AK_with_groups'

    snr_included_ravenpro_human_txt = pd.read_csv(f'{raventxt_human_files_dir}/{raventxt_human_filename}_ravenpro_SNR_aditya_SNR_and_peakfreqtime.txt', sep=sep)
    bd2_human_df_file = snr_included_ravenpro_human_txt.copy()
    bd2_human_df_file = bd2_human_df_file.drop(columns=['Selection', 'View', 'Channel'])
    bd2_human_df_file.rename(columns={'Begin Time (s)':'start_time',
                                'End Time (s)':'end_time',
                                'Low Freq (Hz)':'low_freq',
                                'High Freq (Hz)':'high_freq',
                                'Delta Time (s)':'delta_time_s',
                                'Manually-Verified Phonic Group':'freq_group',
                                'SNR NIST Quick (dB)':'snr_nist_quick_dB'}, inplace=True)
    bd2_human_df_file.sort_values('start_time', inplace=True)

    args = dict()
    args['chunk_size'] = 2
    args['detection_threshold'] = 0.00
    ones = int(args['detection_threshold'])
    decimals = int(int(100*(args['detection_threshold'])) % 100)
    threshold_tag = f"threshold{ones}p{decimals:02}"
    save_loc_reduced_overlaps = Path(f"bd2__{threshold_tag}_chunksize{int(args['chunk_size'])}_{wav_filename}_REDUCED_OVERLAPS.csv")
    filepath_reduced_overlaps = bd2_dets_save_dir / save_loc_reduced_overlaps
    bd2_file_all_dets = pd.read_csv(filepath_reduced_overlaps, sep=',', index_col=0)
    bd2_file_all_dets.rename(columns={'KMEANS_CLASSES':'freq_group'}, inplace=True)

    batdetect2_eval_file = generate_evaluation_df(bd2_human_df_file, bd2_file_all_dets)
    batdetect2_eval_file['input_file_name'] = [f'{wav_filename}.WAV']*len(batdetect2_eval_file)
    batdetect2_eval_file['site_name'] = [SITE_NAMES[site]]*len(batdetect2_eval_file)

    bd2_human_df_file_LF = bd2_human_df_file[bd2_human_df_file['freq_group']=='LF'].copy()
    batdetect2_df_file_LF_reduced_overlaps = bd2_file_all_dets[bd2_file_all_dets['freq_group']=='LF'].copy()
    batdetect2_eval_file_LF = generate_evaluation_df(bd2_human_df_file_LF, batdetect2_df_file_LF_reduced_overlaps)
    batdetect2_eval_file_LF['input_file_name'] = [f'{wav_filename}.WAV']*len(batdetect2_eval_file_LF)
    batdetect2_eval_file_LF['site_name'] = [SITE_NAMES[site]]*len(batdetect2_eval_file_LF)

    bd2_human_df_file_HF = bd2_human_df_file[bd2_human_df_file['freq_group']=='HF'].copy()
    batdetect2_df_file_HF_reduced_overlaps = bd2_file_all_dets[bd2_file_all_dets['freq_group']=='HF'].copy()
    batdetect2_eval_file_HF = generate_evaluation_df(bd2_human_df_file_HF, batdetect2_df_file_HF_reduced_overlaps)
    batdetect2_eval_file_HF['input_file_name'] = [f'{wav_filename}.WAV']*len(batdetect2_eval_file_HF)
    batdetect2_eval_file_HF['site_name'] = [SITE_NAMES[site]]*len(batdetect2_eval_file_HF)

    batdetect2_eval = pd.concat([batdetect2_eval, batdetect2_eval_file])
    batdetect2_eval_LF = pd.concat([batdetect2_eval_LF, batdetect2_eval_file_LF])
    batdetect2_eval_HF = pd.concat([batdetect2_eval_HF, batdetect2_eval_file_HF])

0.0 0.01


In [12]:
batdetect2_eval.to_csv(bd2_dets_save_dir / '20250127__bd2_eval_results_per_thresh_and_file.csv')
batdetect2_eval_LF.to_csv(bd2_dets_save_dir / '20250127__bd2_LFeval_results_per_thresh_and_file.csv')
batdetect2_eval_HF.to_csv(bd2_dets_save_dir / '20250127__bd2_HFeval_results_per_thresh_and_file.csv')